##### Intro to Python Programming
- Prof Joanna Bieri
- Office: Duke 209

### Day 18 - More Advanced Web Scraping 

Today we are going to learn a few different ways to get data from a Website into Python. Web scraping takes quite a bit of patience. Every website is different and they way they store their data is different. So for each site you will have to logically figure out how to access the data.

The Ethical Scraper (from https://towardsdatascience.com/ethics-in-web-scraping-b96b18136f01):

I, the web scraper will live by the following principles:

- If you have a public API that provides the data I’m looking for, I’ll use it and avoid scraping all together.
- I will always provide a User Agent string that makes my intentions clear and provides a way for you to contact me with questions or concerns.
- I will request data at a reasonable rate. I will strive to never be confused for a DDoS attack.
- I will only save the data I absolutely need from your page. If all I need it OpenGraph meta-data, that’s all I’ll keep.
- I will respect any content I do keep. I’ll never pass it off as my own.
- I will look for ways to return value to you. Maybe I can drive some (real) traffic to your site or credit you in an article or post.
- I will respond in a timely fashion to your outreach and work with you towards a resolution.
- I will scrape for the purpose of creating new value from the data, not to duplicate it.

### Go to the website:

https://www.indeed.com/jobs?q=Data+Science&l=Southern+California

### Do you think it is ethical to scrape this data?

### Your answer here:

#
#
#

### See my discussion below

### Is it okay to scrape data from Indeed?

# YES - it is legal but...

There are some issues. First if you have access to data that is not available publicly, for example you have to log in to access that data then YOU SHOULD NOT SCRAPE THAT DATA. Also, indeed is a more complicated site that uses javascript and it has safegaurds against people doing heavy scraping, so YOU SHOULD NOT SRAPE AT RATES THAT COULD DAMAGE THE WEBSITE.

But, as long as you are using the publically available information for personal use OR you are doing something new with the data and giving credit to the source. THEN LETS GET SCRAPING!


### Lets download information from Indeed.com 

Some websites use java script OR have code that recognizes simple data grabs. To deal with things like that we actually need to use a browser to let the code load first. Then you can access the HTML.

In [1]:
import requests
from bs4 import BeautifulSoup

### Here is the code from last time - Read the Text.

In [2]:
website = "https://www.indeed.com/jobs?q=Data+Science&l=Southern+California"
raw_code = requests.get(website)
html_doc = raw_code.text
soup = BeautifulSoup(html_doc, 'html.parser')
soup.get_text()

'Blocked - Indeed.com\n Access DeniedYour request has been blocked.Return home'

### What Happened?

Indeed.com uses javascript when loading it's website AND it watches for people who are trying to scrape the site, forcing them to slow down. Scraping sends requests to websites and if you send too many requests too fast you could drastically slow down or crash the website!

To get around this we use selenium webdriver to open the pages one-by-one allowing them to load before reading the source html. This slows down the requests and is aligned with our ethics of web scraping.

# WARNING - THIS DOES NOT WORK ON COLAB

You will need to download this notebook and run it in jupyter on your computer to access the web browser functionality.

In [3]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from selenium import webdriver

/home/bellajagu/.local/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/bellajagu/.local/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [24]:
# Get the first page:

# Specify the search
area = "Data Science"
location = "Southern+California"
web_address='https://www.indeed.com/jobs?q='+area+'&l='+location

driver = webdriver.Firefox()
driver.get(web_address)

html_source_code = driver.execute_script("return document.body.innerHTML;")

# Save to beautiful soup
soup = BeautifulSoup(html_source_code, 'html.parser')

# Close page
driver.close()

In [25]:
# What is in there?
# You could print it here
# Much better to use developer tools
# soup

### Use developer tools to look at the HTML code:

https://www.indeed.com/jobs?q=Data+Science&l=Southern+California


### Getting Job Names

We can see that the job names are inside a secotion marked h2 with class='jobTitle. We can search for all instances of this in the website.

In [26]:
# Get the names of the jobs
result1 = soup.find_all('h2', class_="jobTitle")
job_names = [r.text for r in result1]
print(len(job_names))
job_names

19


['Data Science Analyst I (Full Time) - United States',
 'Data Science & Analytics',
 'Data Science & Analytics',
 'Data Scientist I',
 'Data Scientist, Handshake AI',
 'Junior Data Engineer',
 'Senior Program Analyst',
 'BI SQL Analytics Developer',
 'Business Analyst',
 'Data Scientist, People Innovation',
 'Data Scientist (Hybrid)',
 'Data Scientist, Preparedness',
 'Data Scientist I',
 'Research Data Analyst',
 'Data Scientist',
 'Data Science Graduate (Advertisement Team) - 2026 Start (BS/MS)',
 'Data Scientist, All Levels',
 'Junior Data Scientist, Stanford Law School',
 'Data Scientist 1']

In [27]:
### We can search for lots of other things:

In [28]:
# Look at the abbreviated job descriptions
result2 = soup.find_all(class_='job_seen_beacon')
descriptions = [r.text for r in result2]
print(len(descriptions))
descriptions

19


['Data Science Analyst I (Full Time) - United StatesCisco SystemsSan Jose, CA$93,700 - $154,300 a yearFull-timePaid parental leaveParental leave401(k)Health insurance401(k) matchingVision insurance',
 'Data Science & AnalyticsBlocksiPalo Alto, CA 94301\xa0(Professorville area)Pay information not provided',
 'Data Science & AnalyticsBlocksiPalo Alto, CA 94301\xa0(Professorville area)Pay information not provided',
 'Data Scientist IInland Empire Health PlanHybrid work in Rancho Cucamonga, CA 91730$104,041.60 - $171,641.60 a yearFull-timeWeekends as needed\u2002+2457(b)Health insuranceVision insuranceDental insuranceFlexible spending accountLife insurance',
 'Data Scientist, Handshake AIEasily applyHandshakeSan Francisco, CA$135,000 - $200,000 a yearFull-timeReferral programPaid parental leaveFood providedParental leaveHealth insurance401(k) matching',
 'Junior Data EngineerWTWCaliforniaPay information not providedFull-timeMilitary leavePrescription drug insuranceAD&D insuranceParental le

In [29]:
# List of indeed links to the named jobs
result3 = soup.find_all('a', class_="jcs-JobTitle")
links = ['https://www.indeed.com' + r.get('href') for r in result3]
print(len(links))
links

19


['https://www.indeed.com/rc/clk?jk=57dc3e930d425754&bb=y5jekCsoJFGkSOY0fm5V0rKzZ4ZdO4pauHos9QAHS_roGbSUzMfrjxYCmEg9GRzQ1jepMYqQg_dVJU-HZpHrZXmL2EFJ7esSvMeJHtrSS6MTLkEMoSriTIDzic_tHI4Jj-xrI32uDJ2_zM1l10XytA%3D%3D&xkcb=SoD167M3m08bpbRWCZ0PbzkdCdPP&fccid=dfc44f3b8c44a6db&vjs=3',
 'https://www.indeed.com/rc/clk?jk=31e696fcb9d9bdcb&bb=y5jekCsoJFGkSOY0fm5V0sXi-mtgf_0gQ7f-IKyWAa-8YHW8uCqaMIYKxWy7BVTfkdsrgaw10Ubypca-afl3FUOwYp7Aq018uIrDzxjvjBwKIUIAYzyYGMPcPRJv9QKaWEb5iAM4Xv6DnzezmRHxqg%3D%3D&xkcb=SoBB67M3m08bpbRWCZ0ObzkdCdPP&fccid=5d3c1614d72915d1&vjs=3',
 'https://www.indeed.com/viewjob?jk=890abcdef0123456',
 'https://www.indeed.com/rc/clk?jk=bd48452205d907f4&bb=y5jekCsoJFGkSOY0fm5V0qZjmVOU6Jll7n8xeq5yXZMgFFsP3tbb7sj29Cm4P2NjWRmNHqmv2wDJFULQY0CD_LboaSMMUDVNpFwT3f1JVgMBsUTpCGKunVEbSdYlpYdB_oxURB7SY0cdjmojhExuLg%3D%3D&xkcb=SoDc67M3m08bpbRWCZ0NbzkdCdPP&fccid=f23cfaf12528dbd0&vjs=3',
 'https://www.indeed.com/rc/clk?jk=1fba899090626bbb&bb=y5jekCsoJFGkSOY0fm5V0kAnGSXhmg8Omtl5loGbMPz0L6umCK-SPJyk4hW

In [31]:
# List of job locations
result4 = soup.find_all(class_="css-1f06pz4 eu4oa1w0")
locations = [r.text for r in result4]
print(len(locations))
locations

19


['San Jose, CA',
 'Palo Alto, CA 94301\xa0(Professorville area)',
 'Palo Alto, CA 94301\xa0(Professorville area)',
 'Hybrid work in Rancho Cucamonga, CA 91730',
 'San Francisco, CA',
 'California',
 'Long Beach, CA 90822',
 'Hybrid work in Los Angeles, CA',
 'Hybrid work in Gold River, CA 95670',
 'San Francisco, CA\xa0(Mission area)',
 'Stanford, CA',
 'San Francisco, CA\xa0(Mission area)',
 'Hybrid work in Santa Monica, CA 90401',
 'Stanford, CA',
 'San Jose, CA 95110\xa0(Downtown area)',
 'San Jose, CA',
 'Hybrid work in San Francisco, CA',
 'Hybrid work in Stanford, CA',
 'Hybrid work in San Jose, CA']

### Beware!

The data is not always available for every instance.

Thics can cause problems if your lists are different lenghts! You might have to write more complicated code to be able to stitch this information together - leaving blank list entries for things that don't have a value on the website. Alternatively you could change your search a bit.

In [33]:
# List of job salaries
result5 = soup.find_all(class_="salary-snippet-container mosaic-provider-jobcards-fswglz e1xnxm2i0")
salaries = [r.text for r in result5]
print(len(salaries))
salaries

16


['$93,700 - $154,300 a year',
 '$104,041.60 - $171,641.60 a year',
 '$135,000 - $200,000 a year',
 '$50 an hour',
 '$105,000 - $125,000 a year',
 '$6,472 - $9,616 a month',
 '$230,000 - $342,000 a year',
 '$99,974 - $113,446 a year',
 '$347,000 - $400,000 a year',
 '$128,000 - $130,000 a year',
 '$80,148 - $99,773 a year',
 '$109,000 - $200,700 a year',
 '$102,600 - $160,000 a year',
 '$137,120 - $297,330 a year',
 '$80,148 - $99,773 a year',
 '$174,304 - $243,500 a year']

### After you have your raw data - Analyze it

I typically put the data into pandas to make it easier to work with.

In [34]:
# Put this info into a pandas data frame:
DF = pd.DataFrame()

DF['Job Names'] = job_names
DF['Location'] = locations
DF['Job Description'] = descriptions
DF['Job Link'] = links

DF

,Job Names,Location,Job Description,Job Link
0,Data Science Analyst I (Full Time) - United St...,"San Jose, CA",Data Science Analyst I (Full Time) - United St...,https://www.indeed.com/rc/clk?jk=57dc3e930d425...
1,Data Science & Analytics,"Palo Alto, CA 94301 (Professorville area)","Data Science & AnalyticsBlocksiPalo Alto, CA 9...",https://www.indeed.com/rc/clk?jk=31e696fcb9d9b...
2,Data Science & Analytics,"Palo Alto, CA 94301 (Professorville area)","Data Science & AnalyticsBlocksiPalo Alto, CA 9...",https://www.indeed.com/viewjob?jk=890abcdef012...
3,Data Scientist I,"Hybrid work in Rancho Cucamonga, CA 91730",Data Scientist IInland Empire Health PlanHybri...,https://www.indeed.com/rc/clk?jk=bd48452205d90...
4,"Data Scientist, Handshake AI","San Francisco, CA","Data Scientist, Handshake AIEasily applyHandsh...",https://www.indeed.com/rc/clk?jk=1fba899090626...
5,Junior Data Engineer,California,Junior Data EngineerWTWCaliforniaPay informati...,https://www.indeed.com/rc/clk?jk=0a179fc6f0cf1...
6,Senior Program Analyst,"Long Beach, CA 90822",Senior Program AnalystEasily applyNewInsync Co...,https://www.indeed.com/pagead/clk?mo=r&ad=-6NY...
7,BI SQL Analytics Developer,"Hybrid work in Los Angeles, CA",BI SQL Analytics DeveloperEasily applyNewMotio...,https://www.indeed.com/pagead/clk?mo=r&ad=-6NY...
8,Business Analyst,"Hybrid work in Gold River, CA 95670",Business AnalystVisions In Education Charter S...,https://www.indeed.com/pagead/clk?mo=r&ad=-6NY...
9,"Data Scientist, People Innovation","San Francisco, CA (Mission area)","Data Scientist, People InnovationOpenAISan Fra...",https://www.indeed.com/rc/clk?jk=d1b5a29dab336...


### Practice 1

Use the code above to look up job information for your area of interest and location. Then look at the html code for the ideed site that you scrape. Choose some new information and see if you can get Python to access it.

Can you get the company info? This is the information just above the location of each job.

In [ ]:
# Your code here:


### Practice 2 - Challenge

Write code using a for loop that opens up each of the links you found on the first page and scrapes information from each of those pages, adding it to the data frame.

In [ ]:
# Your code here:


### Practice 3 - Challenge

Find a website of your choice and see if you can create a pandas data frame based on the information you find there. Make sure you are using good data ethics!

In [ ]:
# Your code here:
